In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install pretty_midi

In [ ]:
import os

folders = [
    '/content/drive/MyDrive/music-project/outputs/generated_midis',
    '/content/drive/MyDrive/music-project/outputs/plots',
    '/content/drive/MyDrive/music-project/outputs/survey_results/tokens',
]
for f in folders:
    os.makedirs(f, exist_ok=True)

print("All folders created!")

All folders created!


In [ ]:
import zipfile, os

zip_path = '/content/drive/MyDrive/Music Project/maestro-v3.0.0-midi.zip'

print("Extracting dataset...")
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/maestro/')


Extracting dataset...


In [ ]:
exec(open('/content/drive/MyDrive/Music Project/task2.py').read())
print("Task 2 code loaded!")

Device       : cpu
Epochs       : 5  (warmup=2)
SEQ_LEN      : 64
Hidden dim   : 128
Max files    : 200 per split

Loading datasets...
[train] 24586 windows | Genre breakdown: {'Modern': 8727, 'Romantic': 15308, 'Baroque': 551}
[validation] 15411 windows | Genre breakdown: {'Modern': 5244, 'Romantic': 8185, 'Baroque': 956, 'Classical': 1026}
Parameters   : 576,856
Epoch  1/5  β=0.00  Train[Total=0.3170 Recon=0.3170 KL=0.8605]  Val[Recon=0.2666]
Epoch  2/5  β=0.00  Train[Total=0.2821 Recon=0.2821 KL=2.0325]  Val[Recon=0.2446]
Epoch  3/5  β=0.33  Train[Total=0.3265 Recon=0.3067 KL=0.0592]  Val[Recon=0.2667]
Epoch  4/5  β=0.67  Train[Total=0.3193 Recon=0.2992 KL=0.0300]  Val[Recon=0.2655]
Epoch  5/5  β=1.00  Train[Total=0.3208 Recon=0.2978 KL=0.0230]  Val[Recon=0.2693]
Loss plots saved: /content/drive/MyDrive/music-project/outputs/plots/loss_curve_task2.png
Model weights saved: /content/drive/MyDrive/music-project/outputs/vae_weights.pt

Generating 8 samples...
  Saved: /content/drive/MyD

In [ ]:
import torch

CSV_PATH  = '/content/maestro/maestro-v3.0.0/maestro-v3.0.0.csv'
MIDI_ROOT = '/content/maestro/maestro-v3.0.0'
OUT_DIR   = '/content/drive/MyDrive/music-project/outputs/generated_midis'
PLOT_DIR  = '/content/drive/MyDrive/music-project/outputs/plots'
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

Device: cpu


In [ ]:
from torch.utils.data import DataLoader

print("Loading dataset...")
train_ds = MAESTROMultiGenreDataset(CSV_PATH, MIDI_ROOT, split='train')
val_ds   = MAESTROMultiGenreDataset(CSV_PATH, MIDI_ROOT, split='validation')

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0)
print("Dataset ready!")

Loading dataset...
[train] 24586 windows | Genre breakdown: {'Modern': 8727, 'Romantic': 15308, 'Baroque': 551}
[validation] 15411 windows | Genre breakdown: {'Modern': 5244, 'Romantic': 8185, 'Baroque': 956, 'Classical': 1026}
Dataset ready!


In [ ]:
LATENT        = 32
EPOCHS        = 5
WARMUP_EPOCHS = 2

model = MusicVAE(input_dim=PIANO_KEYS, hidden_dim=128,
                 latent_dim=LATENT, seq_len=SEQ_LEN)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print("Training...")

history = train(model, train_loader, val_loader,
                epochs=EPOCHS, warmup_epochs=WARMUP_EPOCHS,
                device=DEVICE)

torch.save(model.state_dict(),
           '/content/drive/MyDrive/music-project/outputs/vae_weights.pt')
print("Model saved!")

Parameters: 548,120
Training...
Epoch  1/5  β=0.00  Train[Total=0.3118 Recon=0.3118 KL=1.6898]  Val[Recon=0.2660]
Epoch  2/5  β=0.00  Train[Total=0.2877 Recon=0.2877 KL=2.4810]  Val[Recon=0.2438]
Epoch  3/5  β=0.33  Train[Total=0.3312 Recon=0.3110 KL=0.0606]  Val[Recon=0.2696]
Epoch  4/5  β=0.67  Train[Total=0.3280 Recon=0.3073 KL=0.0311]  Val[Recon=0.2755]
Epoch  5/5  β=1.00  Train[Total=0.3356 Recon=0.3161 KL=0.0195]  Val[Recon=0.2843]
Model saved!


In [ ]:
plot_losses(history, save_path=f'{PLOT_DIR}/loss_curve_task2.png')
print("Plot saved!")

Loss plots saved: /content/drive/MyDrive/music-project/outputs/plots/loss_curve_task2.png
Plot saved!


In [ ]:
generate_samples(model, n=4, threshold=0.2,
                 out_dir=OUT_DIR, device=DEVICE)
print("4 samples done!")

  Saved: /content/drive/MyDrive/music-project/outputs/generated_midis/task2_sample_1.mid  (74 notes)
  Saved: /content/drive/MyDrive/music-project/outputs/generated_midis/task2_sample_2.mid  (85 notes)
  Saved: /content/drive/MyDrive/music-project/outputs/generated_midis/task2_sample_3.mid  (83 notes)
  Saved: /content/drive/MyDrive/music-project/outputs/generated_midis/task2_sample_4.mid  (83 notes)
4 samples done!


In [ ]:
latent_interpolation(model, n_steps=8, threshold=0.2,
                     out_dir=OUT_DIR, device=DEVICE)
print("Interpolation done!")

  Saved: /content/drive/MyDrive/music-project/outputs/generated_midis/task2_interp_1_alpha0.00.mid  (70 notes)
  Saved: /content/drive/MyDrive/music-project/outputs/generated_midis/task2_interp_2_alpha0.14.mid  (71 notes)
  Saved: /content/drive/MyDrive/music-project/outputs/generated_midis/task2_interp_3_alpha0.29.mid  (74 notes)
  Saved: /content/drive/MyDrive/music-project/outputs/generated_midis/task2_interp_4_alpha0.43.mid  (77 notes)
  Saved: /content/drive/MyDrive/music-project/outputs/generated_midis/task2_interp_5_alpha0.57.mid  (79 notes)
  Saved: /content/drive/MyDrive/music-project/outputs/generated_midis/task2_interp_6_alpha0.71.mid  (81 notes)
  Saved: /content/drive/MyDrive/music-project/outputs/generated_midis/task2_interp_7_alpha0.86.mid  (83 notes)
  Saved: /content/drive/MyDrive/music-project/outputs/generated_midis/task2_interp_8_alpha1.00.mid  (84 notes)
Latent interpolation complete.
Interpolation done!


In [ ]:
print("TASK 2 METRICS")
for i in range(1, 9):
    p = f'{OUT_DIR}/task2_sample_{i}.mid'
    if os.path.exists(p):
        rd = rhythm_diversity(p)
        rr = repetition_ratio(p)
        print(f"Sample {i}: Rhythm Div={rd:.3f} | Rep Ratio={rr:.3f}")

TASK 2 METRICS
Sample 1: Rhythm Div=0.027 | Rep Ratio=0.000
Sample 2: Rhythm Div=0.024 | Rep Ratio=0.000
Sample 3: Rhythm Div=0.036 | Rep Ratio=0.000
Sample 4: Rhythm Div=0.036 | Rep Ratio=0.000
Sample 5: Rhythm Div=0.048 | Rep Ratio=0.000
Sample 6: Rhythm Div=0.035 | Rep Ratio=0.000
Sample 7: Rhythm Div=0.065 | Rep Ratio=0.000
Sample 8: Rhythm Div=0.041 | Rep Ratio=0.000
